# Telecom Customer Churn: Exploratory Data Analysis

Starter notebook for validated exploratory analysis. Run `src/clean_data.py` first, then review available columns before drawing conclusions.

## Import Libraries

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_customers.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load Cleaned Dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Run `python src/clean_data.py` from the project root first.")
df = pd.read_csv(DATA_PATH, low_memory=False)
df.head()

## Dataset Overview

In [ ]:
print(f"Shape: {df.shape}")
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))
df.head()

## Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].to_frame("missing_values")

## Churn Distribution

In [ ]:
if "churn_flag" in df.columns:
    display(df["churn_flag"].value_counts(dropna=False).to_frame("customers"))
    sns.countplot(data=df, x="churn_flag")
    plt.title("Customer Churn Distribution")
    plt.show()
else:
    print("churn_flag is not available.")

## Numeric Summaries

In [ ]:
df.select_dtypes(include="number").describe().T

## Categorical Summaries

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "string"]).columns
pd.DataFrame({
    "unique_values": df[categorical_columns].nunique(dropna=False),
    "most_common": [df[col].mode(dropna=True).iloc[0] if not df[col].mode(dropna=True).empty else None for col in categorical_columns],
}).sort_values("unique_values")

## Churn by Contract

In [ ]:
contract_col = next((c for c in ["contract_group", "contract", "contract_type"] if c in df.columns), None)
if contract_col and "churn_flag" in df.columns:
    display(df.groupby(contract_col, dropna=False)["churn_flag"].agg(customers="size", churn_rate="mean"))
else:
    print("Contract and churn fields are required.")

## Churn by Tenure Group

In [ ]:
if {"tenure_group", "churn_flag"}.issubset(df.columns):
    display(df.groupby("tenure_group", dropna=False)["churn_flag"].agg(customers="size", churn_rate="mean"))
else:
    print("tenure_group and churn_flag are required.")

## Churn by Payment Method

In [ ]:
payment_col = next((c for c in ["payment_group", "payment_method", "payment"] if c in df.columns), None)
if payment_col and "churn_flag" in df.columns:
    display(df.groupby(payment_col, dropna=False)["churn_flag"].agg(customers="size", churn_rate="mean"))
else:
    print("Payment and churn fields are required.")

## Churn by Service

In [ ]:
service_columns = [c for c in df.columns if any(term in c for term in ["internet_service", "phone_service", "streaming", "security", "support", "backup"])]
if service_columns and "churn_flag" in df.columns:
    for column in service_columns:
        display(df.groupby(column, dropna=False)["churn_flag"].agg(customers="size", churn_rate="mean").rename_axis(column))
else:
    print("No service columns or churn_flag available.")

## Revenue at Risk

In [ ]:
if "revenue_at_risk" in df.columns:
    print(f"Total revenue at risk: {df['revenue_at_risk'].sum():,.2f}")
    segment_col = next((c for c in ["customer_value_segment", "contract_group", "tenure_group"] if c in df.columns), None)
    if segment_col:
        display(df.groupby(segment_col, dropna=False)["revenue_at_risk"].sum().sort_values(ascending=False))
else:
    print("revenue_at_risk is not available.")

## Export Simple Figures if Needed

In [ ]:
# Example export after validating the visual:
# plt.savefig(FIGURES_DIR / "validated_churn_distribution.png", dpi=150, bbox_inches="tight")